# Gridworld: from logits to a sampled episode

A fixed policy turns the finite-horizon MDP in `envs.py` into trajectories.
We will inspect action probabilities, draw one complete episode, and separate
policy sampling, environmental slip, and reward noise. There is no training here.

All code cells use the Python standard library. Run them in order with the
working directory set to the repository root, `rl_math`, or this notebook's directory.


In [1]:
from pathlib import Path
import math
import platform
import sys

project = next(p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "rl_math")
               if (p / "sampling.py").is_file())
sys.path.insert(0, str(project))
from envs import Action, Gridworld
from returns import discounted_returns
from sampling import TabularSoftmaxPolicy, sample_episode

print("Python", platform.python_version())


Python 3.10.12


## One row of a tabular policy

For the full state $s=(\mathrm{cell},\mathrm{remaining})$, assign four logits
$\theta_{s,a}$ in **north, east, south, west** order:

$$\pi_\theta(a\mid s)=\frac{\exp(\theta_{s,a})}{\sum_b\exp(\theta_{s,b})}.$$

The row $(\log 1,\log 2,\log 3,\log 4)$ gives probabilities
$(0.1,0.2,0.3,0.4)$. `probabilities` subtracts the largest logit before
exponentiating; this leaves the ratios unchanged and avoids overflow.


In [2]:
world = Gridworld()  # Default 5x5 map; horizon=12, slip=0.2.
example = TabularSoftmaxPolicy({world.start_state: tuple(math.log(x) for x in (1, 2, 3, 4))})
for action, probability in zip(Action, example.probabilities(world.start_state)):
    print(f"{action.name:5s}: {probability:.1f}")
assert math.isclose(sum(example.probabilities(world.start_state)), 1)


NORTH: 0.1
EAST : 0.2
SOUTH: 0.3
WEST : 0.4


## A fixed preference for the longer safe route

The bottom row is `SHHHG`. Going north, east four times, then south avoids
the hazards when all moves succeed. Give the preferred action logit $\log 7$
and every other action logit $0$: the policy samples its preference with
probability $0.7$ and each other action with probability $0.1$.

This is a manually specified policy. Every nonterminal cell/time pair has its
own row, although we use the same preference at a cell for each remaining time.
The policy may choose another action, and slip may change where that action
lands. Even the preferred action does not guarantee a safe trajectory.


In [3]:
logits = {}
for state in world.states():
    if world.is_terminal(state):
        continue  # No actions after the goal or the task horizon.
    row, column = state.cell
    preferred = Action.NORTH if row == 4 else Action.EAST if column < 4 else Action.SOUTH
    logits[state] = tuple(math.log(7) if action == preferred else 0.0 for action in Action)
policy = TabularSoftmaxPolicy(logits)
print("Start probabilities:", tuple(round(p, 2) for p in policy.probabilities(world.start_state)))


Start probabilities: (0.7, 0.1, 0.1, 0.1)


## Draw and inspect a complete episode

At each time $t$, `sample_episode` makes three draws:

$$a_t\sim\pi_\theta(\cdot\mid s_t),\qquad
s_{t+1}\sim P(\cdot\mid s_t,a_t),\qquad
r_{t+1}=\bar r(s_t,a_t,s_{t+1})+\epsilon_{t+1},$$

where $\epsilon_{t+1}\sim\mathcal N(0,\sigma^2)$ independently at each step.
Here $\bar r$ is the selected transition's conditional expected reward;
`world.expected_reward(s, a)` additionally averages over possible next states.

Use seed **7**, slip **0.2**, horizon **12**, and reward-noise standard deviation
**2.0** for this illustration. The sampler uses separate named random streams
for actions, transitions, and reward noise and leaves global random state alone.
Repeating a seed replays an episode. Use distinct seeds for new samples, and
keep the Python environment and all settings fixed when reproducing a run.


In [4]:
seed, noise_std, gamma = 7, 2.0, 0.9
episode = sample_episode(world, policy, seed=seed, reward_noise_std=noise_std)
assert episode == sample_episode(world, policy, seed=seed, reward_noise_std=noise_std)
print(f"seed={episode.seed}, sigma={episode.reward_noise_std}, horizon={world.horizon}, slip={world.slip}")
print(" t   cell  left action  reward -> next cell left   end")
for step in episode.steps:
    print(f"{step.t:2d} {str(step.state.cell):>7} {step.state.remaining:4d} {step.action.name:5s} "
          f"{step.reward:7.3f} -> {str(step.next_state.cell):>7} "
          f"{step.next_state.remaining:4d} {str(step.terminated):>5}")
print("Ending:", episode.end_reason)
print(f"G_0 (gamma={gamma}) = {discounted_returns([s.reward for s in episode.steps], gamma)[0]:.3f}")


seed=7, sigma=2.0, horizon=12, slip=0.2
 t   cell  left action  reward -> next cell left   end
 0  (4, 0)   12 NORTH  -3.746 ->  (3, 0)   11 False
 1  (3, 0)   11 EAST    1.594 ->  (3, 1)   10 False
 2  (3, 1)   10 WEST   -1.217 ->  (3, 0)    9 False
 3  (3, 0)    9 EAST   -1.823 ->  (3, 1)    8 False
 4  (3, 1)    8 EAST    3.868 ->  (3, 2)    7 False
 5  (3, 2)    7 WEST   -2.441 ->  (3, 1)    6 False
 6  (3, 1)    6 EAST   -1.126 ->  (3, 2)    5 False
 7  (3, 2)    5 EAST   -0.639 ->  (3, 3)    4 False
 8  (3, 3)    4 EAST   -2.776 ->  (3, 4)    3 False
 9  (3, 4)    3 SOUTH   9.302 ->  (4, 4)    2  True
Ending: goal
G_0 (gamma=0.9) = -2.026


In [5]:
# Mark all visited cells; the table above gives their order and revisits.
visited = {step.state.cell for step in episode.steps} | {episode.steps[-1].next_state.cell}
for row, line in enumerate(world.layout):
    print(" ".join(symbol + ("*" if (row, column) in visited else " ")
                   for column, symbol in enumerate(line)))
print("* visited; S start, G goal, H hazard, # wall")


.  .  .  .  . 
.  .  #  .  . 
.  .  #  .  . 
.* .* .* .* .*
S* H  H  H  G*
* visited; S start, G goal, H hazard, # wall


In the recorded seed-7 run, the policy samples WEST at t=2 and t=5.
Those backtracks come from the policy: each selected action in this
particular path happens to move in its intended direction. Slip can change
the outcome in other episodes. The goal is reached at t=9 with two steps left.

The recorded `t` starts at zero for each episode; it is the exponent in the
outer $\gamma^t$ policy-gradient weight introduced in `returns.py`. Every
actual move decreases remaining time by one, including a move blocked by a wall.

The final transition is marked `terminated=True`: reaching the goal or
exhausting this task's horizon gives zero future value. There is no absorbing
tail, rollout cutoff, or external time-limit truncation in this sampler.
Reaching the goal on the last allowed move still pays the goal reward.


## Why noise changes variance, not the expected reward

The logits change action probabilities and thus which states are visited.
The environment supplies slip and reward noise. With this fixed state-only
policy, reward noise does not influence future actions or transitions.

Independence and zero mean give

$$\mathbb E[r_{t+1}\mid s_t,a_t,s_{t+1}]=\bar r(s_t,a_t,s_{t+1}),$$

so averaging over next states also preserves `world.expected_reward(s, a)`.
The added conditional reward variance is $\sigma^2$. Given a state/action path
of length $T$, the noise contribution to its discounted return has mean zero
and variance

$$\operatorname{Var}\!\left[\sum_{t=0}^{T-1}\gamma^t\epsilon_{t+1}\;\middle|\;
\mathrm{path}\right]=\sigma^2\sum_{t=0}^{T-1}\gamma^{2t}.$$

Unconditionally, expected return is also unchanged; the added return variance
averages this expression over paths. A particular noisy episode can have a
higher or lower return. It need not average to zero within that episode.

Changing only the noise scale keeps the same sampled path because its random
stream is separate. Compare the clean and noisy rewards on that path below.


In [6]:
clean = sample_episode(world, policy, seed=seed)
assert [(s.state, s.action, s.next_state) for s in clean.steps] == [
    (s.state, s.action, s.next_state) for s in episode.steps]
print(" t model reward   noise observed reward")
for base, noisy in zip(clean.steps, episode.steps):
    print(f"{base.t:2d} {base.reward:12.3f} {noisy.reward - base.reward:7.3f} {noisy.reward:15.3f}")
variance = noise_std**2 * sum(gamma**(2 * s.t) for s in episode.steps)
print(f"Added return variance conditional on this path: {variance:.3f}")


 t model reward   noise observed reward
 0       -1.000  -2.746          -3.746
 1       -1.000   2.594           1.594
 2       -1.000  -0.217          -1.217
 3       -1.000  -0.823          -1.823
 4       -1.000   4.868           3.868
 5       -1.000  -1.441          -2.441
 6       -1.000  -0.126          -1.126
 7       -1.000   0.361          -0.639
 8       -1.000  -1.776          -2.776
 9        9.000   0.302           9.302
Added return variance conditional on this path: 18.493
